# 00 — Pre-Flight Verification

**Run this notebook before the workshop.**

If every cell prints `OK` (or a green check), you are set. If any cell errors, see the Troubleshooting section in `SETUP.md`.

This notebook does five checks:

1. Python version is recent enough.
2. All required packages import.
3. The dataset loads.
4. SimPy can build and run a trivial simulation.
5. The Block 5 saved-response fallback file is present.

## 1. Python version

Python 3.10 or newer.

In [7]:
import sys
required = (3, 10)
got = sys.version_info[:3]
print(f"Python {got[0]}.{got[1]}.{got[2]}")
assert got >= required, f"Python {required[0]}.{required[1]}+ required, got {got[0]}.{got[1]}"
print("OK")

Python 3.12.11
OK


## 2. Required packages

Each one prints its version. If any line raises `ModuleNotFoundError`, install with `pip install -r requirements.txt`.

In [8]:
import importlib
required = ["numpy", "scipy", "pandas", "matplotlib", "simpy"]
for name in required:
    mod = importlib.import_module(name)
    print(f"  {name:12s} {mod.__version__}")
print("OK")

  numpy        2.4.4
  scipy        1.17.1
  pandas       2.3.3
  matplotlib   3.10.9
  simpy        4.1.1
OK


## 3. Dataset

The transaction-level POS data ships in `../data/coffee_shop_pos.csv`. We load it and confirm the row count and a few summary statistics.

In [9]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/coffee_shop_pos.csv")
assert DATA_PATH.exists(), f"Dataset not found at {DATA_PATH.resolve()}. See SETUP.md."

df = pd.read_csv(DATA_PATH, parse_dates=["arrival_timestamp"])
print(f"  rows         : {len(df):,}")
print(f"  date range   : {df['arrival_timestamp'].min().date()} → {df['arrival_timestamp'].max().date()}")
print(f"  order types  : {sorted(df['order_type'].unique())}")
print(f"  mean service : {df['service_duration_min'].mean():.2f} min")
assert len(df) > 20000, "Dataset has fewer rows than expected."
assert set(df['order_type'].unique()) == {"drip", "espresso", "blended"}, "Order types do not match the spec."
print("OK")

  rows         : 21,822
  date range   : 2025-10-06 → 2026-04-03
  order types  : ['blended', 'drip', 'espresso']
  mean service : 3.34 min
OK


## 4. SimPy smoke test

A trivial simulation: one server, three customers, deterministic service times. If this runs and prints three departures, SimPy is working.

In [10]:
import simpy

def customer(env, name, server, service_time):
    arrived = env.now
    with server.request() as req:
        yield req
        wait = env.now - arrived
        yield env.timeout(service_time)
        print(f"  {name}: arrived t={arrived:.0f}, waited {wait:.0f}, served for {service_time}")

env = simpy.Environment()
server = simpy.Resource(env, capacity=1)
env.process(customer(env, "C1", server, 3))
env.process(customer(env, "C2", server, 2))
env.process(customer(env, "C3", server, 1))
env.run()
print("OK")

  C1: arrived t=0, waited 0, served for 3
  C2: arrived t=0, waited 3, served for 2
  C3: arrived t=0, waited 5, served for 1
OK


## 5. Block 5 saved-response fallback

The Block 5 notebook calls an LLM if you have a key set, and otherwise falls back to a saved representative response. Confirm the saved response file is present and parseable.

In [11]:
import json
from pathlib import Path

FALLBACK = Path("llm_responses/spec_compliance_review.json")
assert FALLBACK.exists(), f"Saved response not found at {FALLBACK.resolve()}. The Block 5 notebook will fail without it."

review = json.loads(FALLBACK.read_text())
assert "issues" in review, "Saved response missing 'issues' field."
assert len(review["issues"]) >= 2, "Saved response should have at least 2 issues."
print(f"  fallback file: present ({FALLBACK.stat().st_size:,} bytes)")
print(f"  issues       : {len(review['issues'])}")
print(f"  observations : {len(review.get('observations_not_bugs', []))}")
print("OK")

  fallback file: present (5,038 bytes)
  issues       : 2
  observations : 3
OK


## 6. Optional — Anthropic SDK and API key

This check is **optional**. The Block 5 notebook works without it via the saved-response fallback.

If you want to run the live LLM call instead, you need both the `anthropic` package installed and `ANTHROPIC_API_KEY` set in your environment.

In [12]:
import os
key_set = bool(os.environ.get("ANTHROPIC_API_KEY"))

try:
    import anthropic
    sdk_version = anthropic.__version__
    sdk_ok = True
except ImportError:
    sdk_ok = False
    sdk_version = "(not installed)"

print(f"  anthropic SDK    : {sdk_version}")
print(f"  ANTHROPIC_API_KEY: {'set' if key_set else 'not set'}")
print()
if sdk_ok and key_set:
    print("  Live LLM cell available. (Optional — fallback also works.)")
elif sdk_ok and not key_set:
    print("  SDK installed but no API key. Block 5 will use the saved-response fallback.")
else:
    print("  SDK not installed. Block 5 will use the saved-response fallback. This is fine.")

  anthropic SDK    : 0.97.0
  ANTHROPIC_API_KEY: not set

  SDK installed but no API key. Block 5 will use the saved-response fallback.


---

## Done

If every cell above passed, you are ready for the workshop. If any cell failed, check the Troubleshooting section in `SETUP.md` and email dan@sullivanlearninggroup.com if you're stuck.

See you at the conference.